# تدريب نموذج الإنذار المبكر لمخاطر الحضور

هذا الـnotebook يدرب نموذجاً تعليمياً على ملف البيانات **الاصطناعية فقط**. لا يتصل بـNeon، ولا يكتب في قاعدة البيانات، ولا يدمج النموذج بالتطبيق.

الحماية الأساسية: `student_key` و`snapshot_date` والحقول التي تصف نافذة الـlabel المستقبلية مستبعدة من المدخلات. التقسيم يتم حسب الطالب حتى لا تظهر بيانات الطالب نفسه في التدريب والاختبار.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 20_261_023


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'package.json').is_file() and (candidate / 'ml').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from the Ishraf project directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / 'ml' / 'artifacts' / 'student-risk-synthetic' / 'student-risk-synthetic-training.csv'
DATASET_METADATA_PATH = DATA_PATH.with_name('student-risk-synthetic-training.metadata.json')
RUNS_ROOT = PROJECT_ROOT / 'ml' / 'artifacts' / 'training-runs'

assert DATA_PATH.is_file(), f'Missing synthetic dataset: {DATA_PATH}'
assert DATASET_METADATA_PATH.is_file(), f'Missing dataset metadata: {DATASET_METADATA_PATH}'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATA_PATH}')

In [ ]:
dataset_metadata = json.loads(DATASET_METADATA_PATH.read_text(encoding='utf-8'))
assert dataset_metadata['datasetKind'] == 'synthetic', 'This notebook is restricted to synthetic data.'
assert dataset_metadata['realOperationalRecordsIncluded'] is False

data = pd.read_csv(DATA_PATH)
data.shape, dataset_metadata

## عقد التدريب ومنع تسرب البيانات

لا يجوز تدريب النموذج على معرف الطالب أو التاريخ، ولا على عدد ملاحظات الـlabel المستقبلية أو نسبة الحضور المستقبلية. تلك الحقول مفيدة للتدقيق فقط.

In [ ]:
IDENTIFIER_COLUMNS = ['student_key', 'snapshot_date']
LABEL_COLUMN = 'label_at_risk_next_14d'
LABEL_EVIDENCE_COLUMNS = [
    'label_observation_count',
    'label_future_attendance_percent',
]
FORBIDDEN_INPUT_COLUMNS = IDENTIFIER_COLUMNS + [LABEL_COLUMN, *LABEL_EVIDENCE_COLUMNS]
FEATURE_COLUMNS = [column for column in data.columns if column not in FORBIDDEN_INPUT_COLUMNS]

assert FEATURE_COLUMNS
assert not set(FORBIDDEN_INPUT_COLUMNS).intersection(FEATURE_COLUMNS)
assert data['student_key'].nunique() == dataset_metadata['uniqueStudents']
assert set(data[LABEL_COLUMN].dropna().unique()).issubset({True, False})

X = data.loc[:, FEATURE_COLUMNS].copy()
for column in X.select_dtypes(include='bool'):
    X[column] = X[column].astype('int8')
X = X.apply(pd.to_numeric, errors='raise')
y = data[LABEL_COLUMN].astype('int8')
groups = data['student_key']

print(f'Features: {len(FEATURE_COLUMNS)}')
print(f'Positive-label rate: {y.mean():.2%}')
pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'missing_values': X.isna().sum().reindex(FEATURE_COLUMNS).to_numpy(),
    'dtype': X.dtypes.reindex(FEATURE_COLUMNS).astype(str).to_numpy(),
})

In [ ]:
# Use earlier academic periods for training, then one later period each for validation and test.
# A student is assigned by their latest observed period, so their earlier rows are purged
# if they appear in a later holdout. This enforces both chronological and group isolation.
snapshot_dates = pd.to_datetime(data['snapshot_date'], errors='raise')
academic_period = pd.Series(
    np.where(snapshot_dates.dt.month >= 8, snapshot_dates.dt.year, snapshot_dates.dt.year - 1),
    index=data.index,
    name='academic_period',
).astype('int64')
ordered_periods = sorted(pd.unique(academic_period).tolist())
if len(ordered_periods) < 3:
    raise ValueError('At least three academic periods are required for chronological train, validation, and test splits.')

train_periods = ordered_periods[:-2]
validation_period = ordered_periods[-2]
test_period = ordered_periods[-1]
student_last_period = pd.DataFrame({
    'student_key': groups,
    'academic_period': academic_period,
}).groupby('student_key', sort=False)['academic_period'].max()
assigned_period = groups.map(student_last_period).astype('int64')

train_mask = academic_period.isin(train_periods) & assigned_period.isin(train_periods)
validation_mask = academic_period.eq(validation_period) & assigned_period.eq(validation_period)
test_mask = academic_period.eq(test_period) & assigned_period.eq(test_period)
if not train_mask.any() or not validation_mask.any() or not test_mask.any():
    raise ValueError('Chronological split produced an empty partition.')

train_indices = np.flatnonzero(train_mask.to_numpy())
validation_indices = np.flatnonzero(validation_mask.to_numpy())
test_indices = np.flatnonzero(test_mask.to_numpy())
X_train, y_train = X.iloc[train_indices], y.iloc[train_indices]
X_validation, y_validation = X.iloc[validation_indices], y.iloc[validation_indices]
X_test, y_test = X.iloc[test_indices], y.iloc[test_indices]

student_groups = {
    'train': set(groups.iloc[train_indices]),
    'validation': set(groups.iloc[validation_indices]),
    'test': set(groups.iloc[test_indices]),
}
assert not student_groups['train'] & student_groups['validation']
assert not student_groups['train'] & student_groups['test']
assert not student_groups['validation'] & student_groups['test']
assert snapshot_dates.iloc[train_indices].max() < snapshot_dates.iloc[validation_indices].min()
assert snapshot_dates.iloc[validation_indices].max() < snapshot_dates.iloc[test_indices].min()

temporal_split = {
    'period_definition': 'Academic period starts in August.',
    'train_periods': train_periods,
    'validation_period': validation_period,
    'test_period': test_period,
    'purged_rows_for_group_isolation': int((~(train_mask | validation_mask | test_mask)).sum()),
}
split_summary = pd.DataFrame([
    {
        'split': 'train',
        'periods': ','.join(map(str, train_periods)),
        'rows': len(train_indices),
        'students': len(student_groups['train']),
        'positive_rate': y_train.mean(),
        'start_date': snapshot_dates.iloc[train_indices].min().date().isoformat(),
        'end_date': snapshot_dates.iloc[train_indices].max().date().isoformat(),
    },
    {
        'split': 'validation',
        'periods': str(validation_period),
        'rows': len(validation_indices),
        'students': len(student_groups['validation']),
        'positive_rate': y_validation.mean(),
        'start_date': snapshot_dates.iloc[validation_indices].min().date().isoformat(),
        'end_date': snapshot_dates.iloc[validation_indices].max().date().isoformat(),
    },
    {
        'split': 'test',
        'periods': str(test_period),
        'rows': len(test_indices),
        'students': len(student_groups['test']),
        'positive_rate': y_test.mean(),
        'start_date': snapshot_dates.iloc[test_indices].min().date().isoformat(),
        'end_date': snapshot_dates.iloc[test_indices].max().date().isoformat(),
    },
])
split_summary

In [ ]:
def choose_threshold(y_true: pd.Series, probabilities: np.ndarray) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    if len(thresholds) == 0:
        return 0.5

    f1_scores = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    return float(thresholds[int(np.nanargmax(f1_scores))])


def evaluate_classifier(
    model_name: str, y_true: pd.Series, probabilities: np.ndarray, threshold: float
) -> dict[str, float | int | str]:
    predictions = (probabilities >= threshold).astype('int8')
    true_negative, false_positive, false_negative, true_positive = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()
    return {
        'model': model_name,
        'threshold': float(threshold),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'brier_score': float(brier_score_loss(y_true, probabilities)),
        'true_negative': int(true_negative),
        'false_positive': int(false_positive),
        'false_negative': int(false_negative),
        'true_positive': int(true_positive),
    }


models = {
    'logistic_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('classifier', HistGradientBoostingClassifier(
            learning_rate=0.08,
            max_iter=250,
            max_leaf_nodes=24,
            min_samples_leaf=40,
            l2_regularization=0.2,
            random_state=RANDOM_STATE,
        )),
    ]),
}

In [ ]:
validation_reports = []
validation_probabilities: dict[str, np.ndarray] = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_validation)[:, 1]
    threshold = choose_threshold(y_validation, probabilities)
    validation_probabilities[model_name] = probabilities
    validation_reports.append(evaluate_classifier(model_name, y_validation, probabilities, threshold))

validation_results = pd.DataFrame(validation_reports).sort_values('pr_auc', ascending=False)
validation_results

In [ ]:
selected_name = str(validation_results.iloc[0]['model'])
selected_threshold = float(validation_results.iloc[0]['threshold'])
selected_model = models[selected_name]
test_probabilities = selected_model.predict_proba(X_test)[:, 1]
test_report = evaluate_classifier(selected_name, y_test, test_probabilities, selected_threshold)

print(f'Selected model: {selected_name}')
pd.DataFrame([test_report])

In [ ]:
run_id = f'student-risk-synthetic-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}'
run_directory = RUNS_ROOT / run_id
run_directory.mkdir(parents=True, exist_ok=False)

artifact = {
    'model': selected_model,
    'feature_columns': FEATURE_COLUMNS,
    'threshold': selected_threshold,
    'dataset_kind': 'synthetic',
    'scope': 'coursework_only_not_for_production',
}
joblib.dump(artifact, run_directory / 'model.joblib')

report = {
    'run_id': run_id,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'dataset_metadata': dataset_metadata,
    'feature_columns': FEATURE_COLUMNS,
    'temporal_split': temporal_split,
    'split_summary': split_summary.to_dict(orient='records'),
    'validation_results': validation_results.to_dict(orient='records'),
    'test_result': test_report,
    'restrictions': [
        'Synthetic-data metrics are not production performance claims.',
        'Do not integrate this model into Ishraf applications.',
        'Real-data experimentation requires a separate approved grouped and time-aware evaluation.',
    ],
}
(run_directory / 'report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
pd.DataFrame({
    'actual_label': y_test.to_numpy(),
    'probability': test_probabilities,
    'prediction_at_validation_threshold': (test_probabilities >= selected_threshold).astype('int8'),
}).to_csv(run_directory / 'test-predictions.csv', index=False)

print(f'Artifacts saved in: {run_directory}')
report['test_result']

## تفسير النتيجة

هذه النتائج تثبت أن الـpipeline يعمل على بيانات مولدة وفق قواعد معروفة، ولا تثبت أن النموذج صالح لطلاب حقيقيين. لا تنقل النموذج أو الـthreshold إلى أي API أو لوحة تحكم. الانتقال إلى بيانات حقيقية يتطلب بيانات طولية متوازنة ومراجعة خصوصية وتقييم grouped وtime-aware مستقل.